# TCG-pokebot — train, RL, and evaluate on Kaggle (native Linux)

Kaggle is Linux x86-64, so the `cabt` engine loads directly — no Docker.

**Add Input (right panel) before running:**
1. the **Pokémon TCG AI Battle** competition (engine `cg/`, `EN_Card_Data.csv`, sample `deck.csv`)
2. enable **Internet** so cell 1 can `git clone` the code — OR upload the repo as a dataset (cell 1 falls back to that)

Pipeline: locate engine → card tables → import decks → behavioral cloning →
**self-play RL (measured each iteration)** → arena baseline → decide what to submit.

> **Private repo:** Kaggle can't `git clone` it. Run `./pack_for_kaggle.sh`, upload `tcg-pokebot-code.zip` as a Kaggle **Dataset** (new version each code change), and Add Input it. Cell 1 auto-detects the dataset when the clone fails.


In [ ]:
import os, sys, glob, shutil, subprocess

REPO = '/kaggle/working/TCG-pokebot'
def have_code(p): return os.path.exists(os.path.join(p, 'main.py'))

# 1) get our code into a WRITABLE dir (git clone, else fall back to an uploaded dataset)
if not have_code(REPO):
    try:
        subprocess.run(['git','clone','--depth','1',
                        'https://github.com/Hakase-0/TCG-pokebot', REPO], check=True)
    except Exception as e:
        print('git clone failed, looking for an uploaded dataset copy:', e)
        srcs = [g for p in glob.glob('/kaggle/input/*')
                  for g in glob.glob(os.path.join(p, '**', 'main.py'), recursive=True)]
        assert srcs, 'Make the repo public, or upload it as a Kaggle dataset (Add Input), then re-run.'
        shutil.copytree(os.path.dirname(srcs[0]), REPO)
sys.path.insert(0, REPO); os.chdir(REPO)
print('code ready at', REPO)

# 2) locate competition input (engine + card data) and link it into the repo
def _find(name):
    for p in glob.glob('/kaggle/input/*'):
        hits = glob.glob(os.path.join(p, '**', name), recursive=True)
        if hits: return hits[0]
    return None

cg, csv, sample = _find('libcg.so'), _find('EN_Card_Data.csv'), _find('deck.csv')
print('inputs:', glob.glob('/kaggle/input/*'))
if cg and not os.path.exists('cg'):            os.symlink(os.path.dirname(cg), os.path.join(REPO,'cg'))
if csv and not os.path.exists('EN_Card_Data.csv'): os.symlink(csv, os.path.join(REPO,'EN_Card_Data.csv'))
if sample and not os.path.exists('deck.csv'):  shutil.copy(sample, os.path.join(REPO,'deck.csv'))
print('cg:', os.path.exists('cg'), '| csv:', os.path.exists('EN_Card_Data.csv'), '| deck:', os.path.exists('deck.csv'))

get_ipython().system('pip -q install kaggle-environments==1.30.1 >/dev/null 2>&1')
from cg import api; print('engine OK, cards:', len(api.all_card_data()))


In [ ]:
# generate the card capability/attack tables from the engine (one-time)
get_ipython().system('CG_LIB_PATH=$PWD/cg python inspect_cards.py')
print('tables:', os.path.exists('capability_table.json'), os.path.exists('attack_table.json'))


In [ ]:
# refresh meta pool + off-meta ADVERSARIES from Limitless (committed decks/*.txt are the fallback)
import glob, os, subprocess
try:
    subprocess.run(['python','build_deck_pool.py','--top','30','--csv','EN_Card_Data.csv'], check=True)
    subprocess.run(['python','build_deck_pool.py','--archetype','other','--n','10',
                    '--outdir','decks/adversary','--prefix','adv','--csv','EN_Card_Data.csv'], check=True)
except Exception as e:
    print('live refresh skipped, using committed decklists:', e)
from import_deck import import_decklist, write_deck_csv
for txt in glob.glob('decks/*.txt') + glob.glob('decks/adversary/*.txt'):
    csv = txt[:-4] + '.csv'
    if os.path.exists(csv): continue
    ids, rep = import_decklist(open(txt).read(), 'EN_Card_Data.csv')
    if rep['legal_60']: write_deck_csv(ids, csv)
print('meta pool:', len(glob.glob('decks/*.csv')), '| adversaries:', len(glob.glob('decks/adversary/*.csv')))


In [ ]:
# behavioral cloning: distill the combat agent into the net (the RL warm start)
get_ipython().system('python gen_selfplay_data.py --games 300 --policy combat --out data/bc.pkl --log logs/selfplay.jsonl')
get_ipython().system('python train_bc.py --data data/bc.pkl --epochs 20 --out model.pt --log logs/train.jsonl')
import stats; print('BC final:', stats.read('logs/train.jsonl')[-1])


In [ ]:
# ISMCTS PLAY-TIME A/B (Step 2) — does search beat the raw net it wraps? CI tells us if it's real.
# candidate = BC net wrapped in ISMCTS; anchor = raw BC net. Promote/keep line shows winrate + 95% CI.
# Also field-evals the ISMCTS agent vs the pool. (Slow: ISMCTS games are many engine calls each.)
get_ipython().system('python arena.py --candidate model.pt --anchor model.pt --ismcts --ismcts-worlds 4 --ismcts-sims 48 --games 80 --field --log logs/arena.jsonl')
print('Read: vs-anchor winrate >50% with CI low >50% = ISMCTS significantly beats the raw net.')


In [ ]:
# ISMCTS RL RUN (Step 2 training) — visit-count targets + field-gated early-stop. Self-play uses ISMCTS
# (slow); the per-iter field check + gate use the fast raw net. ~10-15 min/iter. Use 'Save & Run All'.
# If field holds/improves vs the BC baseline, the flywheel works -> scale --games/--iters up.
get_ipython().system('python selfplay_rl.py --search ismcts --ismcts-worlds 3 --ismcts-sims 24 --iters 6 --games 120 --gate-games 80 --field-games 5 --warm model.pt --our-deck deck.csv --opp-decks decks/ --out rl_ismcts.pt --log logs/rl_ismcts.jsonl')
import stats
base=[r for r in stats.read('logs/rl_ismcts.jsonl') if r.get('event')=='rl_baseline']
fields=[r for r in stats.read('logs/rl_ismcts.jsonl') if r.get('event')=='rl_field']
print('BC baseline field:', base[-1]['field_winrate'] if base else '?')
print('ISMCTS-RL field per iter:', [r['field_winrate'] for r in fields])
print('gate Elo per iter:', [r.get('elo') for r in stats.read('logs/rl_ismcts.jsonl') if r.get('event')=='gate'])


In [ ]:
# MEASURED BASELINE — is RL better than BC, and which policy should we SUBMIT?
get_ipython().system('python arena.py --candidate rl_model.pt --anchor model.pt --games 200 --field --adversary --log logs/arena.jsonl')

# Deployment question: which DECISION policy pilots our deck best (mirror match)?
import arena, torch
from train_bc import device
dev = device(); db, atk, our, pool = arena._load_ctx()
rl  = arena.load_net('rl_model.pt', dev)
mk_rl_combat   = arena.make_net_agent(rl, dev, db, atk, our, use_combat=True)
mk_heur_combat = arena.make_heuristic_agent(db, atk, our, use_combat=True)
mk_rl_raw      = arena.make_net_agent(rl, dev, db, atk, our, use_combat=False)
print('\n--- mirror: which pilots OUR deck better (100 games each) ---')
for name, A, B in [('RL+combat vs heuristic+combat', mk_rl_combat, mk_heur_combat),
                   ('RL+combat vs RL-raw',          mk_rl_combat, mk_rl_raw)]:
    wa, wb, d = arena.match(A, B, our, our, 100)
    print(f'  {name}: {wa}-{wb}-{d}  ({wa/ max(wa+wb,1):.0%})')
print('\nSUBMIT whichever wins. Default main.py POLICY=heuristic (heuristic+combat) is the')
print('safe ship until RL+combat clearly beats it above.')


In [ ]:
# package the submission. To ship the RL net, set POLICY=nn and use rl_model.pt as model.pt.
import shutil
SUBMIT_NN = False   # <- flip to True ONLY if RL+combat beat heuristic+combat above
if SUBMIT_NN:
    shutil.copy('rl_model.pt', 'model.pt')          # nn policy loads model.pt
    os.environ['POLICY'] = 'nn'
    print('packaging NN (RL) policy')
else:
    print('packaging default heuristic+combat policy (safe baseline)')
get_ipython().system('CG_LIB_PATH=$PWD/cg bash build_submission.sh')
for f in ['model.pt','model_meta.json','rl_model.pt','submission.tar.gz']:
    if os.path.exists(f): shutil.copy(f, '/kaggle/working/'+f)
print('artifacts in /kaggle/working:', [x for x in os.listdir('/kaggle/working') if x.endswith(('.pt','.tar.gz','.json'))])
